<a href="https://colab.research.google.com/github/kitchauhcmus/nlp_multi_intent_and_slot_filling/blob/main/nlp_multi_label_intent_and_slot_filling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/nlp_data_multi_intent_and_slot_filling

/content/drive/MyDrive/nlp_data_multi_intent_and_slot_filling


In [4]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.feature_extraction import DictVectorizer
from scipy.sparse import hstack

print("1. Đang đọc dữ liệu...")
train = pd.read_csv("train.csv").fillna("")
test = pd.read_csv("test.csv").fillna("")

print("2. Đang huấn luyện Intent (Baseline TF-IDF + Logistic Regression)...")
v_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), min_df=2, sublinear_tf=True)
v_word = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)

X_train = hstack([v_char.fit_transform(train["text"]), v_word.fit_transform(train["text"])]).tocsr()
X_test = hstack([v_char.transform(test["text"]), v_word.transform(test["text"])]).tocsr()

mlb = MultiLabelBinarizer()
Y_train = mlb.fit_transform([str(r).split("#") for r in train["intent"]])

clf_int = OneVsRestClassifier(LogisticRegression(solver="liblinear", C=10)).fit(X_train, Y_train)
p = clf_int.predict_proba(X_test)

# Xuất Intent (Ngưỡng 0.5)
test_intents = ["#".join([mlb.classes_[j] for j in np.where(p[i] >= 0.5)[0]] or [mlb.classes_[p[i].argmax()]]) for i in range(len(test))]

print("3. Đang huấn luyện Slot (Baseline Cửa sổ trượt + SVM)...")
def tok_feats(toks, i):
    w = lambda k: toks[i + k] if 0 <= i + k < len(toks) else "<s>"
    return {"w": w(0), "w-1": w(-1), "w+1": w(1), "w-2": w(-2), "w+2": w(2),
            "w-1|w": w(-1) + "|" + w(0), "w|w+1": w(0) + "|" + w(1), "digit": w(0).isdigit()}

feats_train = [tok_feats(r.split(), i) for r in train["text"] for i in range(len(r.split()))]
tags_train = [t for r in train["slots"] for t in str(r).split()]
feats_test = [tok_feats(r.split(), i) for r in test["text"] for i in range(len(r.split()))]

dv = DictVectorizer()
X_slot_train = dv.fit_transform(feats_train)
X_slot_test = dv.transform(feats_test)

clf_slot = SGDClassifier(loss="hinge", alpha=2e-6, max_iter=30).fit(X_slot_train, tags_train)
preds_slot = clf_slot.predict(X_slot_test)

# Nối nhãn Slot
test_slots = []
idx = 0
for text in test["text"]:
    length = len(text.split())
    test_slots.append(" ".join(preds_slot[idx : idx + length]))
    idx += length

# Lưu kết quả Baseline
sub = pd.DataFrame({"id": test["id"], "intent": test_intents, "slots": test_slots})
sub.to_csv("sub_baseline.csv", index=False)
print("=> Đã lưu thành công file: sub_baseline.csv (Khoảng 0.66 điểm tổng)")

1. Đang đọc dữ liệu...
2. Đang huấn luyện Intent (Baseline TF-IDF + Logistic Regression)...
3. Đang huấn luyện Slot (Baseline Cửa sổ trượt + SVM)...
=> Đã lưu thành công file: sub_baseline.csv (Khoảng 0.66 điểm tổng)


In [5]:
!pip install transformers
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer

# --- 1. Cấu hình & Tiền xử lý ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "xlm-roberta-base"
BATCH_SIZE = 16
EPOCHS = 4

train_df = pd.read_csv("train.csv").fillna("")
test_df = pd.read_csv("test.csv").fillna("")

# Lấy từ điển nhãn
mlb = MultiLabelBinarizer()
train_intents = mlb.fit_transform([str(r).split("#") for r in train_df["intent"]])
all_slots = sorted(list(set([s for sublist in train_df["slots"].str.split() for s in sublist])))
slot2id = {s: i for i, s in enumerate(all_slots)}
id2slot = {i: s for s, i in slot2id.items()}

n_int = len(mlb.classes_)
n_slot = len(all_slots)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# --- 2. Xây dựng DataLoader ---
class NLPDataset(Dataset):
    def __init__(self, texts, intents=None, slots=None):
        self.texts = [t.split() for t in texts]
        self.intents = intents
        self.slots = [s.split() for s in slots] if slots is not None else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        words = self.texts[idx]
        encoding = tokenizer(words, is_split_into_words=True, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
        item = {key: val.squeeze(0) for key, val in encoding.items()}

        if self.slots is not None:
            word_ids = encoding.word_ids(batch_index=0)
            slot_ids = []
            prev_word_idx = None
            for word_idx in word_ids:
                if word_idx is None or word_idx == prev_word_idx:
                    slot_ids.append(-100) # Ignored tokens
                else:
                    slot_ids.append(slot2id[self.slots[idx][word_idx]])
                prev_word_idx = word_idx
            item["labels_slot"] = torch.tensor(slot_ids, dtype=torch.long)
            item["labels_int"] = torch.tensor(self.intents[idx], dtype=torch.float)

        # Lưu lại word_ids để dùng lúc dự đoán
        item["word_ids"] = str(encoding.word_ids(batch_index=0))
        return item

train_ds = NLPDataset(train_df["text"], train_intents, train_df["slots"])
test_ds = NLPDataset(test_df["text"])
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE)

# --- 3. Kiến trúc Mô hình (Đúng chuẩn Slide) ---
class Joint(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = AutoModel.from_pretrained(MODEL_NAME)
        h = self.enc.config.hidden_size
        self.int_head = nn.Linear(h, n_int)
        self.slot_head = nn.Linear(h, n_slot)

    def forward(self, input_ids, attention_mask):
        H = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.int_head(H[:, 0]), self.slot_head(H)

model = Joint().to(DEVICE)
bce, ce = nn.BCEWithLogitsLoss(), nn.CrossEntropyLoss(ignore_index=-100)

# Cấu hình Learning Rate phân hóa
heads = list(model.int_head.parameters()) + list(model.slot_head.parameters())
opt = torch.optim.AdamW([
    {"params": model.enc.parameters(), "lr": 3e-5},
    {"params": heads, "lr": 1e-3}
], weight_decay=0.01)

# --- 4. Huấn luyện thực tế (Training Loop) ---
print("Bắt đầu huấn luyện XLM-R (Chạy 4 Epochs)...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        opt.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        y_int = batch["labels_int"].to(DEVICE)
        y_slot = batch["labels_slot"].to(DEVICE)

        li, ls = model(input_ids, attention_mask)
        # Cộng loss trực tiếp, cân bằng sức mạnh bằng LR (Thí nghiệm 4)
        loss = bce(li, y_int) + ce(ls.view(-1, n_slot), y_slot.view(-1))

        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Loss Epoch {epoch+1}: {total_loss/len(train_dl):.4f}")

# --- 5. Dự đoán trên tập Test ---
print("Đang chạy dự đoán và xuất file Slot...")
model.eval()
test_slots_pred = []

with torch.no_grad():
    for batch in test_dl:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        _, ls = model(input_ids, attention_mask)

        preds = torch.argmax(ls, dim=2).cpu().numpy()

        # Khôi phục nhãn theo đúng số lượng token gốc của câu
        for i in range(len(preds)):
            word_ids = eval(batch["word_ids"][i])
            pred_sequence = preds[i]

            final_slots = []
            prev_word_idx = None
            for word_idx, pred_idx in zip(word_ids, pred_sequence):
                if word_idx is not None and word_idx != prev_word_idx:
                    final_slots.append(id2slot[pred_idx])
                prev_word_idx = word_idx
            test_slots_pred.append(" ".join(final_slots))

sub_xlmr = pd.DataFrame({"id": test_df["id"], "intent": "N/A", "slots": test_slots_pred})
sub_xlmr.to_csv("sub_xlmr.csv", index=False)
print("=> Đã lưu thành công file: sub_xlmr.csv")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

...

Loss Epoch 2: 0.3189


Epoch 3/4: 100%|██████████| 887/887 [06:21<00:00,  2.33it/s]


Loss Epoch 3: 0.2415


Epoch 4/4: 100%|██████████| 887/887 [06:20<00:00,  2.33it/s]


Loss Epoch 4: 0.1884
Đang chạy dự đoán và xuất file Slot...
=> Đã lưu thành công file: sub_xlmr.csv


In [6]:
import pandas as pd

print("Đang đọc kết quả từ các mô hình...")
base = pd.read_csv("sub_baseline.csv")
xlmr = pd.read_csv("sub_xlmr.csv")

print("Tiến hành ghép: Intent (Baseline 0.77) + Slot (XLM-R 0.72)...")
sub = base[["id", "intent"]].merge(xlmr[["id", "slots"]], on="id")

# Double check độ dài để tránh tạch nộp bài do mất dòng
assert len(sub) == len(base) == len(xlmr), "Lỗi: Số lượng dòng bị rớt trong quá trình merge!"

sub.to_csv("sub_ghep.csv", index=False)
print("Tất cả đã hoàn tất! File nộp bài của bạn là: sub_ghep.csv")
print("=> Tải file này trên Drive về máy và nộp ngay lên hệ thống để gặt 0.75 điểm nhé!")

Đang đọc kết quả từ các mô hình...
Tiến hành ghép: Intent (Baseline 0.77) + Slot (XLM-R 0.72)...
Tất cả đã hoàn tất! File nộp bài của bạn là: sub_ghep.csv
=> Tải file này trên Drive về máy và nộp ngay lên hệ thống để gặt 0.75 điểm nhé!
